# FIT TRADIZIONALE con multi-start Monte Carlo


---

Fit di tutti gli spettri usando le classi [PoissonNoise](/ml_odmr/PoissonNoise.py) e [FitMonteCarlo](/ml_odmr/FitMonteCarlo.py).

In [1]:
import os
import time
from datetime import datetime
from itertools import product
import h5py
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from ml_odmr.FitMonteCarlo import FitMonteCarlo
from ml_odmr.PoissonNoise import PoissonNoise

# =========================================================================
# 1. PARAMETRI DELLA GRIGLIA DA ESEGUIRE STANOTTE
# =========================================================================
list_contrast = [0.8, 0.5, 0.2, 0.1, 0.08, 0.05, 0.02, 0.01]
list_Nph = [3e5, 6e5, 12e5, 24e5, 48e5, 60e5]
list_mc_trials = [1, 5, 10, 20]#, 40]

mat_file_path = "../data/10k/odmr_easyspin_raw_10000.mat"
# Percorso nella root del workspace (allo stesso livello di 'data/')
base_output_dir = os.path.abspath("../benchmarks/10k")
os.makedirs(base_output_dir, exist_ok=True)

random_seed = 42

# Caricamento dati dal file .mat
with h5py.File(mat_file_path, "r") as f:
    spectra_data = np.array(f["spectra_data"]).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()
    labels_c = np.array(f["labels_c"]).squeeze()
    labels_s = np.array(f["labels_s"]).squeeze()

f_ghz = np.asarray(mw_freqs, dtype=np.float64)
N_spectra_to_run = len(spectra_data)

noise_gen = PoissonNoise(seed=random_seed)


# =========================================================================
# 2. FUNZIONE WORKER PER SINGOLO SPETTRO
# =========================================================================
def worker_fit(idx: int, contrast_val: float, n_ph_val: float, m_trials: int) -> dict:
    y_raw = spectra_data[idx]
    
    ideal_sig = noise_gen.ideal_signal(y_raw, contrast=contrast_val)
    noisy_sig = noise_gen.add_noise(
        ideal_sig, 
        n_tot=n_ph_val, 
        seed_modifier=idx
    )
    
    fitter = FitMonteCarlo(M=m_trials)
    res = fitter.fit_single(f_ghz, noisy_sig)
    
    c_true = float(labels_c[idx])
    s_true = float(labels_s[idx])
    
    err_c = abs(res.c_norm - c_true) if res.converged else np.nan
    err_s = abs(res.s_norm - s_true) if res.converged else np.nan
    
    return {
        "spectrum_id": idx,
        "converged": res.converged,
        "c_true_norm": c_true,
        "s_true_norm": s_true,
        "c_fit_norm": res.c_norm,
        "s_fit_norm": res.s_norm,
        "err_c_norm": err_c,
        "err_s_norm": err_s,
        "c_fit_mhz": res.c_mhz,
        "s_fit_mhz": res.s_mhz,
        "chi2_red": res.chi2_red,
        "fit_time_sec": res.duration_sec
    }


# =========================================================================
# 3. LOOP NOTTURNO CON STRUTTURA A SOTTOCARTELLE
# =========================================================================
all_combinations = list(product(list_mc_trials, list_contrast, list_Nph))
total_combos = len(all_combinations)

print(f"Totale combinazioni nella griglia: {total_combos}")
print(f"Spettri per combinazione: {N_spectra_to_run}")
print(f"Cartella principale nel workspace: '{base_output_dir}'\n")

t_night_start = time.time()

with Parallel(n_jobs=-1, verbose=0) as parallel:
    
    for combo_idx, (m_val, c_val, n_ph) in enumerate(all_combinations, start=1):
        
        # 1. Salta le combinazioni già eseguite (C=0.8, M=20, N_ph < 60e5)
        if np.isclose(c_val, 0.8) and m_val == 20 and n_ph < 60e5:
            print(f"[{combo_idx}/{total_combos}] SALTO (già fatto): C={c_val}, N_ph={n_ph:.0e}, M={m_val}")
            continue
            
        # 2. Costruzione gerarchia sottocartelle:
        # benchmarks/10k/contrast=c_val/mc_trials=m_val/
        sub_dir = os.path.join(
            base_output_dir, 
            f"contrast={c_val}", 
            f"mc_trials={m_val}"
        )
        os.makedirs(sub_dir, exist_ok=True)
        
        # Nome del file dentro la sottocartella
        filename = f"fit_mc_Nph{int(n_ph)}.csv"
        filepath = os.path.join(sub_dir, filename)
        
        # 3. Salta se il file esiste già su disco
        if os.path.exists(filepath):
            print(f"[{combo_idx}/{total_combos}] SALTO (file esistente): contrast={c_val}/mc_trials={m_val}/{filename}")
            continue
            
        print(f"[{combo_idx}/{total_combos}] Avvio: C={c_val:<4} | M={m_val:<2} | N_ph={n_ph:.1e} ...", end=" ", flush=True)
        t_combo_start = time.time()
        
        results_dicts = parallel(
            delayed(worker_fit)(i, c_val, n_ph, m_val) for i in range(N_spectra_to_run)
        )
        
        combo_wall_time = time.time() - t_combo_start
        
        df_res = pd.DataFrame(results_dicts)
        n_conv = int(df_res["converged"].sum())
        succ_rate = (n_conv / N_spectra_to_run) * 100.0
        mae_c = df_res["err_c_norm"].mean()
        mae_s = df_res["err_s_norm"].mean()
        mean_t = df_res["fit_time_sec"].mean()
        
        header_metadata = (
            f"# ODMR Monte Carlo Fit Benchmark\n"
            f"# Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"# Source File: {os.path.basename(mat_file_path)}\n"
            f"# N_spectra: {N_spectra_to_run}\n"
            f"# N_photons: {n_ph:.2e}\n"
            f"# Contrast: {c_val}\n"
            f"# MC_Trials_M: {m_val}\n"
            f"# Base_Seed: {random_seed}\n"
            f"# Converged_Spectra: {n_conv}/{N_spectra_to_run} ({succ_rate:.2f}%)\n"
            f"# Mean_MAE_c_norm: {mae_c:.6f}\n"
            f"# Mean_MAE_s_norm: {mae_s:.6f}\n"
            f"# Mean_Fit_Time_Per_Spectrum_s: {mean_t:.4f}\n"
            f"# Total_Wall_Time_s: {combo_wall_time:.2f}\n"
        )
        
        with open(filepath, mode="w", encoding="utf-8") as f:
            f.write(header_metadata)
            df_res.to_csv(f, index=False, float_format="%.6f")
            
        elapsed_total_min = (time.time() - t_night_start) / 60.0
        print(f"Fatto in {combo_wall_time:.1f}s | Conv: {succ_rate:.1f}% | MAE_s: {mae_s:.4f} | (Tot: {elapsed_total_min:.1f} min)")

print(f"\n🎉 Batch notturno completato in {(time.time() - t_night_start)/3600:.2f} ore!")

Totale combinazioni nella griglia: 192
Spettri per combinazione: 10000
Cartella principale nel workspace: '/Users/zosojack/ml-odmr/benchmarks/10k'

[1/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph300000.csv
[2/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph600000.csv
[3/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph1200000.csv
[4/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph2400000.csv
[5/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph4800000.csv
[6/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph6000000.csv
[7/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph300000.csv
[8/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph600000.csv
[9/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph1200000.csv
[10/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph2400000.csv
[11/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_

---

## **PRIMA RUN**: con trials da 1 a 40

In [2]:
"""
Totale combinazioni nella griglia: 240
Spettri per combinazione: 10000
Cartella principale nel workspace: '/Users/zosojack/ml-odmr/benchmarks/10k'

[1/240] Avvio: C=0.8  | M=1  | N_ph=3.0e+05 ... Fatto in 25.9s | Conv: 100.0% | MAE_s: 0.0759 | (Tot: 0.4 min)
[2/240] Avvio: C=0.8  | M=5  | N_ph=3.0e+05 ... Fatto in 115.9s | Conv: 100.0% | MAE_s: 0.0127 | (Tot: 2.4 min)
[3/240] Avvio: C=0.8  | M=10 | N_ph=3.0e+05 ... Fatto in 259.0s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 6.7 min)
[4/240] SALTO (già fatto): C=0.8, N_ph=3e+05, M=20
[5/240] Avvio: C=0.8  | M=40 | N_ph=3.0e+05 ... Fatto in 1189.3s | Conv: 100.0% | MAE_s: 0.0003 | (Tot: 26.5 min)
[6/240] Avvio: C=0.8  | M=1  | N_ph=6.0e+05 ... Fatto in 22.4s | Conv: 100.0% | MAE_s: 0.0760 | (Tot: 26.9 min)
[7/240] Avvio: C=0.8  | M=5  | N_ph=6.0e+05 ... Fatto in 107.3s | Conv: 100.0% | MAE_s: 0.0119 | (Tot: 28.7 min)
[8/240] Avvio: C=0.8  | M=10 | N_ph=6.0e+05 ... Fatto in 210.4s | Conv: 100.0% | MAE_s: 0.0016 | (Tot: 32.2 min)
[9/240] SALTO (già fatto): C=0.8, N_ph=6e+05, M=20
[10/240] Avvio: C=0.8  | M=40 | N_ph=6.0e+05 ... Fatto in 851.8s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 46.4 min)
[11/240] Avvio: C=0.8  | M=1  | N_ph=1.2e+06 ... Fatto in 18.5s | Conv: 100.0% | MAE_s: 0.0759 | (Tot: 46.7 min)
[12/240] Avvio: C=0.8  | M=5  | N_ph=1.2e+06 ... Fatto in 85.1s | Conv: 100.0% | MAE_s: 0.0127 | (Tot: 48.1 min)
[13/240] Avvio: C=0.8  | M=10 | N_ph=1.2e+06 ... Fatto in 175.0s | Conv: 100.0% | MAE_s: 0.0014 | (Tot: 51.0 min)
[14/240] SALTO (già fatto): C=0.8, N_ph=1e+06, M=20
[15/240] Avvio: C=0.8  | M=40 | N_ph=1.2e+06 ... Fatto in 678.8s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 62.3 min)
[16/240] Avvio: C=0.8  | M=1  | N_ph=2.4e+06 ... Fatto in 17.1s | Conv: 100.0% | MAE_s: 0.0783 | (Tot: 62.6 min)
[17/240] Avvio: C=0.8  | M=5  | N_ph=2.4e+06 ... Fatto in 78.4s | Conv: 100.0% | MAE_s: 0.0132 | (Tot: 63.9 min)
[18/240] Avvio: C=0.8  | M=10 | N_ph=2.4e+06 ... Fatto in 156.3s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 66.5 min)
[19/240] SALTO (già fatto): C=0.8, N_ph=2e+06, M=20
[20/240] Avvio: C=0.8  | M=40 | N_ph=2.4e+06 ... Fatto in 621.2s | Conv: 100.0% | MAE_s: 0.0001 | (Tot: 76.9 min)
[21/240] Avvio: C=0.8  | M=1  | N_ph=4.8e+06 ... Fatto in 16.4s | Conv: 100.0% | MAE_s: 0.0798 | (Tot: 77.2 min)
[22/240] Avvio: C=0.8  | M=5  | N_ph=4.8e+06 ... Fatto in 75.9s | Conv: 100.0% | MAE_s: 0.0150 | (Tot: 78.4 min)
[23/240] Avvio: C=0.8  | M=10 | N_ph=4.8e+06 ... Fatto in 152.0s | Conv: 100.0% | MAE_s: 0.0019 | (Tot: 81.0 min)
[24/240] SALTO (già fatto): C=0.8, N_ph=5e+06, M=20
[25/240] Avvio: C=0.8  | M=40 | N_ph=4.8e+06 ... Fatto in 602.7s | Conv: 100.0% | MAE_s: 0.0001 | (Tot: 91.0 min)
[26/240] Avvio: C=0.8  | M=1  | N_ph=6.0e+06 ... Fatto in 16.2s | Conv: 100.0% | MAE_s: 0.0824 | (Tot: 91.3 min)
[27/240] Avvio: C=0.8  | M=5  | N_ph=6.0e+06 ... Fatto in 76.1s | Conv: 100.0% | MAE_s: 0.0141 | (Tot: 92.6 min)
[28/240] Avvio: C=0.8  | M=10 | N_ph=6.0e+06 ... Fatto in 151.8s | Conv: 100.0% | MAE_s: 0.0023 | (Tot: 95.1 min)
[29/240] Avvio: C=0.8  | M=20 | N_ph=6.0e+06 ... Fatto in 301.4s | Conv: 100.0% | MAE_s: 0.0001 | (Tot: 100.1 min)
[30/240] Avvio: C=0.8  | M=40 | N_ph=6.0e+06 ... Fatto in 604.6s | Conv: 100.0% | MAE_s: 0.0001 | (Tot: 110.2 min)
[31/240] Avvio: C=0.5  | M=1  | N_ph=3.0e+05 ... Fatto in 56.2s | Conv: 99.7% | MAE_s: 0.0755 | (Tot: 111.1 min)
[32/240] Avvio: C=0.5  | M=5  | N_ph=3.0e+05 ... Fatto in 273.3s | Conv: 100.0% | MAE_s: 0.0128 | (Tot: 115.7 min)
[33/240] Avvio: C=0.5  | M=10 | N_ph=3.0e+05 ... Fatto in 554.9s | Conv: 100.0% | MAE_s: 0.0018 | (Tot: 125.0 min)
[34/240] Avvio: C=0.5  | M=20 | N_ph=3.0e+05 ... Fatto in 1222.9s | Conv: 100.0% | MAE_s: 0.0007 | (Tot: 145.3 min)
[35/240] Avvio: C=0.5  | M=40 | N_ph=3.0e+05 ... Fatto in 2543.5s | Conv: 100.0% | MAE_s: 0.0007 | (Tot: 187.7 min)
[36/240] Avvio: C=0.5  | M=1  | N_ph=6.0e+05 ... Fatto in 49.7s | Conv: 99.9% | MAE_s: 0.0764 | (Tot: 188.6 min)
[37/240] Avvio: C=0.5  | M=5  | N_ph=6.0e+05 ... Fatto in 220.2s | Conv: 100.0% | MAE_s: 0.0121 | (Tot: 192.2 min)
[38/240] Avvio: C=0.5  | M=10 | N_ph=6.0e+05 ... Fatto in 428.5s | Conv: 100.0% | MAE_s: 0.0021 | (Tot: 199.4 min)
[39/240] Avvio: C=0.5  | M=20 | N_ph=6.0e+05 ... Fatto in 874.8s | Conv: 100.0% | MAE_s: 0.0005 | (Tot: 214.0 min)
[40/240] Avvio: C=0.5  | M=40 | N_ph=6.0e+05 ... Fatto in 1744.1s | Conv: 100.0% | MAE_s: 0.0005 | (Tot: 243.0 min)
[41/240] Avvio: C=0.5  | M=1  | N_ph=1.2e+06 ... Fatto in 30.0s | Conv: 100.0% | MAE_s: 0.0762 | (Tot: 243.5 min)
[42/240] Avvio: C=0.5  | M=5  | N_ph=1.2e+06 ... Fatto in 146.3s | Conv: 100.0% | MAE_s: 0.0123 | (Tot: 246.0 min)
[43/240] Avvio: C=0.5  | M=10 | N_ph=1.2e+06 ... Fatto in 296.5s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 250.9 min)
[44/240] Avvio: C=0.5  | M=20 | N_ph=1.2e+06 ... Fatto in 588.4s | Conv: 100.0% | MAE_s: 0.0004 | (Tot: 260.7 min)
[45/240] Avvio: C=0.5  | M=40 | N_ph=1.2e+06 ... Fatto in 1173.9s | Conv: 100.0% | MAE_s: 0.0003 | (Tot: 280.3 min)
[46/240] Avvio: C=0.5  | M=1  | N_ph=2.4e+06 ... Fatto in 22.8s | Conv: 100.0% | MAE_s: 0.0771 | (Tot: 280.7 min)
[47/240] Avvio: C=0.5  | M=5  | N_ph=2.4e+06 ... Fatto in 107.2s | Conv: 100.0% | MAE_s: 0.0137 | (Tot: 282.5 min)
[48/240] Avvio: C=0.5  | M=10 | N_ph=2.4e+06 ... Fatto in 220.5s | Conv: 100.0% | MAE_s: 0.0019 | (Tot: 286.1 min)
[49/240] Avvio: C=0.5  | M=20 | N_ph=2.4e+06 ... Fatto in 438.1s | Conv: 100.0% | MAE_s: 0.0003 | (Tot: 293.4 min)
[50/240] Avvio: C=0.5  | M=40 | N_ph=2.4e+06 ... Fatto in 863.0s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 307.8 min)
[51/240] Avvio: C=0.5  | M=1  | N_ph=4.8e+06 ... Fatto in 20.3s | Conv: 100.0% | MAE_s: 0.0759 | (Tot: 308.2 min)
[52/240] Avvio: C=0.5  | M=5  | N_ph=4.8e+06 ... Fatto in 94.1s | Conv: 100.0% | MAE_s: 0.0128 | (Tot: 309.7 min)
[53/240] Avvio: C=0.5  | M=10 | N_ph=4.8e+06 ... Fatto in 186.3s | Conv: 100.0% | MAE_s: 0.0021 | (Tot: 312.8 min)
[54/240] Avvio: C=0.5  | M=20 | N_ph=4.8e+06 ... Fatto in 372.6s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 319.1 min)
[55/240] Avvio: C=0.5  | M=40 | N_ph=4.8e+06 ... Fatto in 744.6s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 331.5 min)
[56/240] Avvio: C=0.5  | M=1  | N_ph=6.0e+06 ... Fatto in 20.4s | Conv: 100.0% | MAE_s: 0.0780 | (Tot: 331.8 min)
[57/240] Avvio: C=0.5  | M=5  | N_ph=6.0e+06 ... Fatto in 91.5s | Conv: 100.0% | MAE_s: 0.0140 | (Tot: 333.3 min)
[58/240] Avvio: C=0.5  | M=10 | N_ph=6.0e+06 ... Fatto in 183.6s | Conv: 100.0% | MAE_s: 0.0018 | (Tot: 336.4 min)
[59/240] Avvio: C=0.5  | M=20 | N_ph=6.0e+06 ... Fatto in 365.9s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 342.5 min)
[60/240] Avvio: C=0.5  | M=40 | N_ph=6.0e+06 ... Fatto in 726.9s | Conv: 100.0% | MAE_s: 0.0002 | (Tot: 354.6 min)
[61/240] Avvio: C=0.2  | M=1  | N_ph=3.0e+05 ... Fatto in 101.7s | Conv: 99.0% | MAE_s: 0.0784 | (Tot: 356.3 min)
[62/240] Avvio: C=0.2  | M=5  | N_ph=3.0e+05 ... Fatto in 505.8s | Conv: 100.0% | MAE_s: 0.0113 | (Tot: 364.8 min)
[63/240] Avvio: C=0.2  | M=10 | N_ph=3.0e+05 ... Fatto in 1001.4s | Conv: 100.0% | MAE_s: 0.0029 | (Tot: 381.4 min)
[64/240] Avvio: C=0.2  | M=20 | N_ph=3.0e+05 ... Fatto in 1959.4s | Conv: 100.0% | MAE_s: 0.0021 | (Tot: 414.1 min)
[65/240] Avvio: C=0.2  | M=40 | N_ph=3.0e+05 ... Fatto in 3954.4s | Conv: 100.0% | MAE_s: 0.0021 | (Tot: 480.0 min)
[66/240] Avvio: C=0.2  | M=1  | N_ph=6.0e+05 ... Fatto in 96.0s | Conv: 99.1% | MAE_s: 0.0785 | (Tot: 481.6 min)
[67/240] Avvio: C=0.2  | M=5  | N_ph=6.0e+05 ... Fatto in 469.3s | Conv: 100.0% | MAE_s: 0.0114 | (Tot: 489.4 min)
[68/240] Avvio: C=0.2  | M=10 | N_ph=6.0e+05 ... Fatto in 935.7s | Conv: 100.0% | MAE_s: 0.0023 | (Tot: 505.0 min)
[69/240] Avvio: C=0.2  | M=20 | N_ph=6.0e+05 ... Fatto in 1830.1s | Conv: 100.0% | MAE_s: 0.0015 | (Tot: 535.5 min)
[70/240] Avvio: C=0.2  | M=40 | N_ph=6.0e+05 ... Fatto in 3465.3s | Conv: 100.0% | MAE_s: 0.0014 | (Tot: 593.3 min)
[71/240] Avvio: C=0.2  | M=1  | N_ph=1.2e+06 ... Fatto in 74.1s | Conv: 99.4% | MAE_s: 0.0770 | (Tot: 594.5 min)
[72/240] Avvio: C=0.2  | M=5  | N_ph=1.2e+06 ... Fatto in 354.6s | Conv: 100.0% | MAE_s: 0.0118 | (Tot: 600.4 min)
[73/240] Avvio: C=0.2  | M=10 | N_ph=1.2e+06 ... Fatto in 715.3s | Conv: 100.0% | MAE_s: 0.0023 | (Tot: 612.4 min)
[74/240] Avvio: C=0.2  | M=20 | N_ph=1.2e+06 ... Fatto in 1412.1s | Conv: 100.0% | MAE_s: 0.0012 | (Tot: 635.9 min)
[75/240] Avvio: C=0.2  | M=40 | N_ph=1.2e+06 ...
"""

"\nTotale combinazioni nella griglia: 240\nSpettri per combinazione: 10000\nCartella principale nel workspace: '/Users/zosojack/ml-odmr/benchmarks/10k'\n\n[1/240] Avvio: C=0.8  | M=1  | N_ph=3.0e+05 ... Fatto in 25.9s | Conv: 100.0% | MAE_s: 0.0759 | (Tot: 0.4 min)\n[2/240] Avvio: C=0.8  | M=5  | N_ph=3.0e+05 ... Fatto in 115.9s | Conv: 100.0% | MAE_s: 0.0127 | (Tot: 2.4 min)\n[3/240] Avvio: C=0.8  | M=10 | N_ph=3.0e+05 ... Fatto in 259.0s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 6.7 min)\n[4/240] SALTO (già fatto): C=0.8, N_ph=3e+05, M=20\n[5/240] Avvio: C=0.8  | M=40 | N_ph=3.0e+05 ... Fatto in 1189.3s | Conv: 100.0% | MAE_s: 0.0003 | (Tot: 26.5 min)\n[6/240] Avvio: C=0.8  | M=1  | N_ph=6.0e+05 ... Fatto in 22.4s | Conv: 100.0% | MAE_s: 0.0760 | (Tot: 26.9 min)\n[7/240] Avvio: C=0.8  | M=5  | N_ph=6.0e+05 ... Fatto in 107.3s | Conv: 100.0% | MAE_s: 0.0119 | (Tot: 28.7 min)\n[8/240] Avvio: C=0.8  | M=10 | N_ph=6.0e+05 ... Fatto in 210.4s | Conv: 100.0% | MAE_s: 0.0016 | (Tot: 32.2 min)\

---

## **SECONDA RUN**: trials da 1 a 20

20 trials impiegano mezz'ora!

In [3]:
"""
Totale combinazioni nella griglia: 192
Spettri per combinazione: 10000
Cartella principale nel workspace: '/Users/zosojack/ml-odmr/benchmarks/10k'

[1/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph300000.csv
[2/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph300000.csv
[3/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph300000.csv
[4/192] SALTO (già fatto): C=0.8, N_ph=3e+05, M=20
[5/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph600000.csv
[6/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph600000.csv
[7/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph600000.csv
[8/192] SALTO (già fatto): C=0.8, N_ph=6e+05, M=20
[9/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph1200000.csv
[10/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph1200000.csv
[11/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph1200000.csv
[12/192] SALTO (già fatto): C=0.8, N_ph=1e+06, M=20
[13/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph2400000.csv
[14/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph2400000.csv
[15/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph2400000.csv
[16/192] SALTO (già fatto): C=0.8, N_ph=2e+06, M=20
[17/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph4800000.csv
[18/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph4800000.csv
[19/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph4800000.csv
[20/192] SALTO (già fatto): C=0.8, N_ph=5e+06, M=20
[21/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph6000000.csv
[22/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph6000000.csv
[23/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph6000000.csv
[24/192] SALTO (file esistente): contrast=0.8/mc_trials=20/fit_mc_Nph6000000.csv
[25/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph300000.csv
[26/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph300000.csv
[27/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph300000.csv
[28/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph300000.csv
[29/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph600000.csv
[30/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph600000.csv
[31/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph600000.csv
[32/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph600000.csv
[33/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph1200000.csv
[34/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph1200000.csv
[35/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph1200000.csv
[36/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph1200000.csv
[37/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph2400000.csv
[38/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph2400000.csv
[39/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph2400000.csv
[40/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph2400000.csv
[41/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph4800000.csv
[42/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph4800000.csv
[43/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph4800000.csv
[44/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph4800000.csv
[45/192] SALTO (file esistente): contrast=0.5/mc_trials=1/fit_mc_Nph6000000.csv
[46/192] SALTO (file esistente): contrast=0.5/mc_trials=5/fit_mc_Nph6000000.csv
[47/192] SALTO (file esistente): contrast=0.5/mc_trials=10/fit_mc_Nph6000000.csv
[48/192] SALTO (file esistente): contrast=0.5/mc_trials=20/fit_mc_Nph6000000.csv
[49/192] SALTO (file esistente): contrast=0.2/mc_trials=1/fit_mc_Nph300000.csv
[50/192] SALTO (file esistente): contrast=0.2/mc_trials=5/fit_mc_Nph300000.csv
[51/192] SALTO (file esistente): contrast=0.2/mc_trials=10/fit_mc_Nph300000.csv
[52/192] SALTO (file esistente): contrast=0.2/mc_trials=20/fit_mc_Nph300000.csv
[53/192] SALTO (file esistente): contrast=0.2/mc_trials=1/fit_mc_Nph600000.csv
[54/192] SALTO (file esistente): contrast=0.2/mc_trials=5/fit_mc_Nph600000.csv
[55/192] SALTO (file esistente): contrast=0.2/mc_trials=10/fit_mc_Nph600000.csv
[56/192] SALTO (file esistente): contrast=0.2/mc_trials=20/fit_mc_Nph600000.csv
[57/192] SALTO (file esistente): contrast=0.2/mc_trials=1/fit_mc_Nph1200000.csv
[58/192] SALTO (file esistente): contrast=0.2/mc_trials=5/fit_mc_Nph1200000.csv
[59/192] SALTO (file esistente): contrast=0.2/mc_trials=10/fit_mc_Nph1200000.csv
[60/192] SALTO (file esistente): contrast=0.2/mc_trials=20/fit_mc_Nph1200000.csv
[61/192] Avvio: C=0.2  | M=1  | N_ph=2.4e+06 ... Fatto in 54.4s | Conv: 99.8% | MAE_s: 0.0768 | (Tot: 0.9 min)
[62/192] Avvio: C=0.2  | M=5  | N_ph=2.4e+06 ... Fatto in 251.1s | Conv: 100.0% | MAE_s: 0.0127 | (Tot: 5.1 min)
[63/192] Avvio: C=0.2  | M=10 | N_ph=2.4e+06 ... Fatto in 521.3s | Conv: 100.0% | MAE_s: 0.0019 | (Tot: 13.8 min)
[64/192] Avvio: C=0.2  | M=20 | N_ph=2.4e+06 ... Fatto in 1047.7s | Conv: 100.0% | MAE_s: 0.0008 | (Tot: 31.3 min)
[65/192] Avvio: C=0.2  | M=1  | N_ph=4.8e+06 ... Fatto in 37.9s | Conv: 100.0% | MAE_s: 0.0751 | (Tot: 31.9 min)
[66/192] Avvio: C=0.2  | M=5  | N_ph=4.8e+06 ... Fatto in 181.0s | Conv: 100.0% | MAE_s: 0.0128 | (Tot: 34.9 min)
[67/192] Avvio: C=0.2  | M=10 | N_ph=4.8e+06 ... Fatto in 358.1s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 40.9 min)
[68/192] Avvio: C=0.2  | M=20 | N_ph=4.8e+06 ... Fatto in 715.1s | Conv: 100.0% | MAE_s: 0.0007 | (Tot: 52.8 min)
[69/192] Avvio: C=0.2  | M=1  | N_ph=6.0e+06 ... Fatto in 31.8s | Conv: 100.0% | MAE_s: 0.0749 | (Tot: 53.3 min)
[70/192] Avvio: C=0.2  | M=5  | N_ph=6.0e+06 ... Fatto in 155.0s | Conv: 100.0% | MAE_s: 0.0132 | (Tot: 55.9 min)
[71/192] Avvio: C=0.2  | M=10 | N_ph=6.0e+06 ... Fatto in 319.4s | Conv: 100.0% | MAE_s: 0.0022 | (Tot: 61.2 min)
[72/192] Avvio: C=0.2  | M=20 | N_ph=6.0e+06 ... Fatto in 629.2s | Conv: 100.0% | MAE_s: 0.0005 | (Tot: 71.7 min)
[73/192] Avvio: C=0.1  | M=1  | N_ph=3.0e+05 ... Fatto in 112.9s | Conv: 99.0% | MAE_s: 0.0783 | (Tot: 73.6 min)
[74/192] Avvio: C=0.1  | M=5  | N_ph=3.0e+05 ... Fatto in 564.3s | Conv: 100.0% | MAE_s: 0.0144 | (Tot: 83.0 min)
[75/192] Avvio: C=0.1  | M=10 | N_ph=3.0e+05 ... Fatto in 1136.6s | Conv: 100.0% | MAE_s: 0.0061 | (Tot: 102.0 min)
[76/192] Avvio: C=0.1  | M=20 | N_ph=3.0e+05 ... Fatto in 2254.8s | Conv: 100.0% | MAE_s: 0.0056 | (Tot: 139.5 min)
[77/192] Avvio: C=0.1  | M=1  | N_ph=6.0e+05 ... Fatto in 100.7s | Conv: 99.0% | MAE_s: 0.0776 | (Tot: 141.2 min)
[78/192] Avvio: C=0.1  | M=5  | N_ph=6.0e+05 ... Fatto in 494.5s | Conv: 100.0% | MAE_s: 0.0109 | (Tot: 149.5 min)
[79/192] Avvio: C=0.1  | M=10 | N_ph=6.0e+05 ... Fatto in 998.0s | Conv: 100.0% | MAE_s: 0.0042 | (Tot: 166.1 min)
[80/192] Avvio: C=0.1  | M=20 | N_ph=6.0e+05 ... Fatto in 1960.4s | Conv: 100.0% | MAE_s: 0.0032 | (Tot: 198.8 min)
[81/192] Avvio: C=0.1  | M=1  | N_ph=1.2e+06 ... Fatto in 86.3s | Conv: 98.9% | MAE_s: 0.0754 | (Tot: 200.2 min)
[82/192] Avvio: C=0.1  | M=5  | N_ph=1.2e+06 ... Fatto in 426.6s | Conv: 100.0% | MAE_s: 0.0116 | (Tot: 207.3 min)
[83/192] Avvio: C=0.1  | M=10 | N_ph=1.2e+06 ... Fatto in 870.2s | Conv: 100.0% | MAE_s: 0.0029 | (Tot: 221.8 min)
[84/192] Avvio: C=0.1  | M=20 | N_ph=1.2e+06 ... Fatto in 1743.1s | Conv: 100.0% | MAE_s: 0.0022 | (Tot: 250.9 min)
[85/192] Avvio: C=0.1  | M=1  | N_ph=2.4e+06 ... Fatto in 84.3s | Conv: 99.1% | MAE_s: 0.0763 | (Tot: 252.3 min)
[86/192] Avvio: C=0.1  | M=5  | N_ph=2.4e+06 ... Fatto in 402.7s | Conv: 100.0% | MAE_s: 0.0117 | (Tot: 259.0 min)
[87/192] Avvio: C=0.1  | M=10 | N_ph=2.4e+06 ... Fatto in 826.2s | Conv: 100.0% | MAE_s: 0.0024 | (Tot: 272.8 min)
[88/192] Avvio: C=0.1  | M=20 | N_ph=2.4e+06 ... Fatto in 1636.4s | Conv: 100.0% | MAE_s: 0.0015 | (Tot: 300.1 min)
[89/192] Avvio: C=0.1  | M=1  | N_ph=4.8e+06 ... Fatto in 73.6s | Conv: 99.4% | MAE_s: 0.0787 | (Tot: 301.3 min)
[90/192] Avvio: C=0.1  | M=5  | N_ph=4.8e+06 ... Fatto in 343.0s | Conv: 100.0% | MAE_s: 0.0130 | (Tot: 307.0 min)
[91/192] Avvio: C=0.1  | M=10 | N_ph=4.8e+06 ... Fatto in 707.6s | Conv: 100.0% | MAE_s: 0.0025 | (Tot: 318.8 min)
[92/192] Avvio: C=0.1  | M=20 | N_ph=4.8e+06 ... Fatto in 1395.7s | Conv: 100.0% | MAE_s: 0.0012 | (Tot: 342.1 min)
[93/192] Avvio: C=0.1  | M=1  | N_ph=6.0e+06 ... Fatto in 64.4s | Conv: 99.5% | MAE_s: 0.0749 | (Tot: 343.1 min)
[94/192] Avvio: C=0.1  | M=5  | N_ph=6.0e+06 ... Fatto in 295.1s | Conv: 100.0% | MAE_s: 0.0110 | (Tot: 348.1 min)
[95/192] Avvio: C=0.1  | M=10 | N_ph=6.0e+06 ... Fatto in 614.9s | Conv: 100.0% | MAE_s: 0.0020 | (Tot: 358.3 min)
[96/192] Avvio: C=0.1  | M=20 | N_ph=6.0e+06 ... Fatto in 1071.4s | Conv: 100.0% | MAE_s: 0.0010 | (Tot: 376.2 min)
[97/192] Avvio: C=0.08 | M=1  | N_ph=3.0e+05 ... Fatto in 97.9s | Conv: 98.3% | MAE_s: 0.0810 | (Tot: 377.8 min)
[98/192] Avvio: C=0.08 | M=5  | N_ph=3.0e+05 ... Fatto in 497.8s | Conv: 100.0% | MAE_s: 0.0174 | (Tot: 386.1 min)
[99/192] Avvio: C=0.08 | M=10 | N_ph=3.0e+05 ... 
"""

"\nTotale combinazioni nella griglia: 192\nSpettri per combinazione: 10000\nCartella principale nel workspace: '/Users/zosojack/ml-odmr/benchmarks/10k'\n\n[1/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph300000.csv\n[2/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph300000.csv\n[3/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph300000.csv\n[4/192] SALTO (già fatto): C=0.8, N_ph=3e+05, M=20\n[5/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph600000.csv\n[6/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph600000.csv\n[7/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph600000.csv\n[8/192] SALTO (già fatto): C=0.8, N_ph=6e+05, M=20\n[9/192] SALTO (file esistente): contrast=0.8/mc_trials=1/fit_mc_Nph1200000.csv\n[10/192] SALTO (file esistente): contrast=0.8/mc_trials=5/fit_mc_Nph1200000.csv\n[11/192] SALTO (file esistente): contrast=0.8/mc_trials=10/fit_mc_Nph1200000.csv\n[12/192] SALTO (già fa